The objective of part 4 is to test the scale invariance of $\Delta B _s$ at different s (scale) values, by plotting the different normalized pdf followed by the sample at varying $s$ values

Get the data:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sp

filename='Y1.txt'
data = pd.read_csv(filename).to_numpy()
B = np.cumsum(data)
s_values = 2**np.array([3,4,5,6,7,8,9])
deltaB = [] # empty list, to be filled with s values
df = pd.DataFrame()

def poly_fit(x, y, order=0):
    # revert order of coeffs from x^0 to x^3
    coeff = np.polyfit(x, y, order)
    p = np.poly1d(coeff)
    return p(x)

def get_deviation(y, s):
    #preparo array deviations
    deviations = np.zeros(len(y))

    x = np.arange(len(y))
    segment_length = 2*s
    # numero di segmenti su cui fare sliding
     
    for k in range(0, len(y), segment_length):
        # prendo un segmento e faccio fit
        x_slice = x[k:k+segment_length]
        y_slice = y[k:k+segment_length]
        y_fitted = poly_fit(x_slice, y_slice, order=3)
        #plt.plot(x_slice, y_slice)
        #plt.plot(x_slice, y_fitted)
        dev = y_slice-y_fitted
        
        deviations[k:k+segment_length] = dev
    return deviations
for s in s_values:
    z = get_deviation(B, s)
    deltaB.append(z[0:1000])

    df[s] = z

## Testing the scale invariance of the process:

To test the scale invariance of  $\Delta B _s$, we started by normalizing the pdfs obtained from the dataset at different $s$ values.


After having obtained $\Delta B _s(i) = B^*(i+s)- B^*(i)$ , we calculated $\sigma_s$ for each dataset, and we computed the pdfs of $\Delta B _s$ from the histogram over their distribution. To normalize these distribution and check the scale invariance, we did the variable change 

$\Delta B _s \rightarrow \Delta B _s/\sigma_s$

and having to maintain the integral to one,

$y = P ( \Delta B _s ) \rightarrow y' = P ( \Delta B _s ) \cdot \sigma_s $

, since

$
\int^{+\infty}_{-\infty}P(\Delta B_s)\ dB_s = 1 = \int^{+\infty}_{-\infty} ( P(\Delta B_s)\cdot \sigma_s )\  dB_s/\sigma_s
$




## Non-Gaussian fit: The Castaing's equation

In order to obtain an appropriate fit for the fat tails observed in the dataset, we tried fitting with the Castaing's equation:

$\tilde{P_s}(x) = \int P_L(x/\sigma)\frac{1}{\sigma}G_{s,L}(ln\sigma)d(ln\sigma)$

Here, the PDF at scale s , $\tilde{P_s}(x)$, is considered as a mixture of gaussians with different $\sigma$ values. In particular:

- $ P_L(x/\sigma)\frac{1}{\sigma}$ is a standard gaussian, the limit distribution of the increments.
- $G_{s,L}(ln\sigma)$ is the distribution of the standard deviations, assuming it i s a gaussian in $ln\sigma$ ($\sigma$ Log-Normal) as in the paper.

Since the fitted pdf is standardized, $E[\sigma^2]=1$ (1), defining $\xi = ln\sigma$ and assuming  $\xi\sim N(\mu,\lambda^2)$, from (1) we get $\mu = -\lambda^2$. 

So, we only have a free parameter $\lambda^2$, and $G_{s,L}$ assumes the form:

$G_{s,L}(ln\sigma) = \frac{1}{\sqrt{2\pi}\lambda}\text{exp}\big(-\frac{(ln\sigma+\lambda^2)^2}{2\lambda^2}\big)$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.integrate import cumulative_trapezoid


def castaing_integrand(ln_sigma, x, lambda2):
    '''
    Funzione usata per valutare la pdf di casting
    '''
    mu = -lambda2 
    sigma_val = np.exp(ln_sigma)
    
    # P_L: Gaussiana standard (media 0, dev std 1)
    p_l = (1 / (np.sqrt(2 * np.pi) * sigma_val)) * np.exp(-0.5 * (x / sigma_val)**2)
    
    # G: Kernel Gaussiano per ln(sigma) con varianza lambda2
    g_kernel = (1 / (np.sqrt(2 * np.pi * lambda2))) * np.exp(-0.5 * (ln_sigma - mu)**2 / lambda2)
    
    return p_l * g_kernel

def castaing_pdf(x, lambda2):
    '''
        Funzione per ottenere la pdf di casting
        x = dataset
        lambda2 = parametro
    '''
    # mi assicuro di avere un np array
    x = np.atleast_1d(x)
    
    # Definizione della griglia di integrazione per u = ln(sigma)

    sigma_u = np.sqrt(lambda2)
    mu = -lambda2  
    limit = 6.0 * sigma_u 
    
    u = np.linspace(mu - limit, mu + limit, 150)
    du = u[1] - u[0]
    u = u[None, :]   # Aggiungo asse per broadcasting
    
    # Calcolo dell'Integranda vettorializzato
    # L'integrale è: P(x) = Integral [ P_L(x/sigma) * (1/sigma) * G(u) ] du
    # Dove sigma = exp(u)    

    sigma_grid = np.exp(u)
    
    # Kernel Log-Normale G(u) (Dipende solo da u e lambda2)
    # G(u) = (1 / sqrt(2*pi*lambda2)) * exp( - (u - mu)^2 / (2*lambda2) )

    norm_G = 1.0 / (np.sqrt(2 * np.pi) * sigma_u)
    G_u = norm_G * np.exp(- (u - mu)**2 / (2 * lambda2))
    
    # Parte Gaussiana P_L(x/sigma) * (1/sigma)
    # x deve essere colonna (N x 1) per broadcastare contro u (1 x M) -> Matrice (N x M)
    X = x[:, None] 
    
    # P_L Normale Standard N(0,1)
    # P_L(z) = (1/sqrt(2pi)) * exp(-z^2/2) con z = x/sigma
    norm_P = 1.0 / np.sqrt(2 * np.pi)
    term_P = (norm_P / sigma_grid) * np.exp(- (X / sigma_grid)**2 / 2.0)
    
    # Integrazione (Regola dei Trapezi lungo l'asse u)
    # Moltiplichiamo i termini e integriamo
    integrand = term_P * G_u
    
    # Somma lungo l'asse 1 (l'asse di u) * passo du
    y_vals = np.trapezoid(integrand, dx=du, axis=1)
    
    if x.size == 1:
        return y_vals[0]
    return y_vals

def castaing_cdf(x, lambda2):
    """
    Calcola la CDF interpolando su una griglia fissa.
    """

    sigma_proxy = np.sqrt(lambda2) if lambda2 > 0 else 1.0
    limit = 100 * sigma_proxy  

    grid_x = np.linspace(-limit, limit, 5000)
    
    # Valuta la PDF sulla griglia (Vettorizzato)
    pdf_vals = castaing_pdf(grid_x, lambda2)
    
    # Integrazione Numerica Cumulativa
    cdf_grid = cumulative_trapezoid(pdf_vals, grid_x, initial=0)
    
    # ultimo 1.0
    cdf_grid /= cdf_grid[-1]
    
    # Interpolazione lineare
    return np.interp(x, grid_x, cdf_grid)

def normalize_ds(data):
    '''
        Normalizza i dati per il plot scale
    '''
    sigma = data.std()
    nbins = int(np.sqrt(len(data)))
    pdf, bin_edges = np.histogram(data-data.mean(), bins=nbins, density=True)

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    x = bin_centers / sigma
    y =  pdf * sigma
    return(x,y)

def anderson_darling_statistic(data, cdf_function, args=()):
    """
    Calcola la statistica A^2 di Anderson-Darling per una distribuzione generica.
    
    Parameters:
    - data: array dei dati osservati (i tuoi incrementi)
    - cdf_function: la tua funzione CDF della Castaing
    - args: parametri della tua funzione
    """
    n = len(data)
    data = np.sort(data)
    
    # Calcola la CDF teorica per ogni punto dei dati osservati
    # u_i è la probabilità teorica di osservare quel dato
    u = cdf_function(data, *args)
    
    # per evitare log(0)
    u = np.clip(u, 1e-10, 1 - 1e-10)
    
    # 2. Formula di Anderson-Darling
    # S = sum( (2i - 1) * [ln(u_i) + ln(1 - u_{n+1-i})] )
    idx = np.arange(1, n + 1)
    S = np.sum((2 * idx - 1) * (np.log(u) + np.log(1 - u[::-1])))
    
    A2 = -n - S / n
    return A2

def rvs_castaing(lambda2, size=1):
    """
    Genera campioni casuali dalla distribuzione di Castaing.
    Logica:
    1. u ~ N(-lambda^2, lambda^2)  -> log-varianza
    2. sigma = exp(u)              -> deviazione standard locale
    3. x ~ N(0, sigma^2)           -> incremento finale
    """
    mu_u = -lambda2
    sigma_u = np.sqrt(lambda2)
    
    # Campioniamo le varianze logaritmiche
    u = np.random.normal(loc=mu_u, scale=sigma_u, size=size)
    sigmas = np.exp(u)
    
    # Campioniamo x condizionato a sigma (x = sigma * Gaussiana(0,1))
    z = np.random.normal(loc=0, scale=1, size=size)
    x = sigmas * z
    return x

def bootstrap_ad_test(observed_A2, best_lambda, n_samples_data, n_simulations=100):
    """
    Esegue il test Monte Carlo per calcolare il p-value.
    
    Args:
    - observed_A2: il valore A^2 calcolato sui tuoi dati veri
    - best_lambda: il lambda stimato sui tuoi dati veri
    - n_samples_data: lunghezza del tuo dataset (len(data_s))
    - n_simulations: quanti dataset sintetici generare (consigliato >= 100)
    """
    sim_A2_scores = []
       
    for i in range(n_simulations):
        # Generiamo dati sintetici con il lambda osservato
        synthetic_data = rvs_castaing(best_lambda, size=n_samples_data)
        # Normalizzo
        synth_mean = np.mean(synthetic_data)
        synth_std = np.std(synthetic_data)
        synthetic_data_norm = (synthetic_data - synth_mean) / synth_std
        
        # Ottengo l'hist
        bin_centers_s, pdf_s = normalize_ds(synthetic_data) 
        
        try:
            # Fit
            p0 = [best_lambda]
            bounds = (0.001, 2.0)
            popt, _ = curve_fit(castaing_pdf, bin_centers_s, pdf_s, p0=p0, bounds=bounds)
            sim_lambda = popt[0]
            
            # Calcolo A^2 sintetico con il lambda appena stimato
            sim_score = anderson_darling_statistic(synthetic_data_norm, castaing_cdf, args=(sim_lambda,))
            sim_A2_scores.append(sim_score)
            
        except Exception as e:
            continue # Se il fit fallisce, saltiamo l'iterazione
            
    sim_A2_scores = np.array(sim_A2_scores)
    
    # Calcolo P-value: Frazione di score simulati maggiori di quello osservato
    p_value = np.mean(sim_A2_scores > observed_A2)
    
    return p_value, sim_A2_scores

def bootstrap_ad_test_subsampled(data_real, best_lambda, n_subsample=500, n_simulations=100):
    """
    Esegue il test AD su sottocampioni dei dati per evitare che N troppo grande
    faccia fallire il test per deviazioni trascurabili.
    """
    # Prendo un campione di lunghezza scelta dai miei dati

    subset_indices = np.random.choice(len(data_real), n_subsample, replace=False)
    data_subset = data_real[subset_indices]
    
    # Normalizzazione del sottocampione reale
    data_subset = (data_subset - np.mean(data_subset)) / np.std(data_subset)
    
    # Score osservato (sul subset)
    observed_A2 = anderson_darling_statistic(data_subset, castaing_cdf, args=(best_lambda,))
    
    sim_A2_scores = []
    
    for i in range(n_simulations):
        # Genero sottocampione sintetico
        synthetic_data = rvs_castaing(best_lambda, size=n_subsample)
        # Normalizzo
        syn_norm = (synthetic_data - np.mean(synthetic_data)) / np.std(synthetic_data)
        
        #uso per velocità del ciclo lo stesso lambda da cui sono estratti dati
        sim_score = anderson_darling_statistic(syn_norm, castaing_cdf, args=(best_lambda,))
        sim_A2_scores.append(sim_score)
    
    sim_A2_scores = np.array(sim_A2_scores)
    
    # P-value
    p_value = np.mean(sim_A2_scores > observed_A2)
    
    return p_value, observed_A2, np.mean(sim_A2_scores)

def get_res(df, Anderson = True):
    # Dizionari per salvare i risultati
    results = {} # bin_centers e pdf per ogni s
    lambdas = [] # valori di lambda2
    lambda_errors = [] # errori
    anderson_scores = {} # Dizionario per gli score AD
    for s in df.columns:

        data_s = df[s].dropna().values

        # PDF sperimentale
        bin_centers, pdf = normalize_ds(data_s)
        
        # Parametri iniziali e bounds
        p0 = [0.1]
        bounds = (0.001, 2.0)
        
        try:
            # fit
            popt, pcov = curve_fit(castaing_pdf, bin_centers, pdf, p0=p0, bounds=bounds)
            
            l2 = popt[0]
            l2_err = np.sqrt(np.diag(pcov))[0]
            
            # Salvataggio risultati
            lambdas.append(l2)
            lambda_errors.append(l2_err)
            results[s] = (bin_centers, pdf)
            
            print(f"Scale s={s}: Lambda^2 = {l2:.3f} +/- {l2_err:.3f}")
            
            # aggiungo lo score di anderson
            if Anderson:
                sigma_s = np.std(data_s)
                mean_s = np.mean(data_s)
                data = (data_s - mean_s) / sigma_s
                score = anderson_darling_statistic(data, castaing_cdf, args=(l2,))
                anderson_scores[s] = score
                print(f"   -> Anderson Score: {score:.4f}")
                p_val, sim_scores = bootstrap_ad_test(score, l2, len(data_s), n_simulations=200)
                
                data_clean = data_s[~np.isnan(data_s)]

                p_val, obs_a2, mean_sim_a2 = bootstrap_ad_test_subsampled(
                    data_clean, 
                    l2,
                    n_subsample=500, 
                    n_simulations=100
                )

                print(f"Scale {s}: P-value={p_val:.3f} (Obs A2={obs_a2:.2f}, Mean Sim A2={mean_sim_a2:.2f})")
                
        except Exception as e:
            print(f"Fitting error for scale {s}: {e}")

    return(results,lambdas,lambda_errors,anderson_scores)

def plot_results(results, lambdas, lambda_errors, show_collapse=True, show_lambda=True):
    """
    Plotta i risultati dell'analisi di invarianza di scala.
    
    Args:
        results: Dizionario con i dati per ogni scala.
        lambdas: Lista dei valori lambda^2.
        lambda_errors: Lista degli errori su lambda^2.
        show_collapse (bool): Se True, mostra il Collapse Plot (sinistra).
        show_lambda (bool): Se True, mostra il grafico Lambda vs Log(s) (destra).
    """
    
    # Media Pesata
    weights = 1 / np.array(lambda_errors)**2
    lambda_mean = np.average(lambdas, weights=weights)
    err_mean = 1 / np.sqrt(np.sum(weights))

    print(f"W. Mean Lambda^2 +/- err: {lambda_mean:.4f} +/- {err_mean:.4f}")

    # Se non è richiesto nessun plot, esci
    if not show_collapse and not show_lambda:
        return

    # 2. Configurazione dinamica della figura
    if show_collapse and show_lambda:
        fig, (ax1, ax2) = plt.subplots(figsize=(16, 6), nrows=1, ncols=2)
    elif show_collapse:
        fig, ax1 = plt.subplots(figsize=(8, 6))
        ax2 = None
    elif show_lambda:
        fig, ax2 = plt.subplots(figsize=(8, 6))
        ax1 = None

    # A: Collapse Plot (Scale Invariance)
    if show_collapse and ax1 is not None:
        colors = plt.colormaps['viridis'](np.linspace(0, 1, len(results)))

        for i, s in enumerate(results):
            bc, p = results[s]
            ax1.semilogy(bc, p, 'o', markersize=3, alpha=0.5, color=colors[i], label=f's={s}')

        # Curva teorica
        x_theory = np.linspace(-5, 5, 100)
        y_theory = castaing_pdf(x_theory, lambda_mean)
        
        ax1.semilogy(x_theory, y_theory, 'r-', linewidth=2.5, label=rf'Global Fit $\lambda^2={lambda_mean:.2f}$')
        ax1.set_title('Collapse Plot: Critical Scale Invariance')
        ax1.set_xlabel(r'$\Delta_s B / \sigma$')
        ax1.set_ylabel(r'$\sigma_s \cdot P(\Delta_s B)$')
        ax1.set_ylim(5*1e-4, 2)
        
        
        if len(results) > 10:
            # Mostra solo fit globale nella legenda se ci sono troppe voci
            handles, labels = ax1.get_legend_handles_labels()
            ax1.legend([handles[-1]], [labels[-1]], loc='best')
        else:
            ax1.legend(loc='best', fontsize='small')
            
        ax1.grid(True, which="both", ls="-", alpha=0.3)

    # B: Lambda vs Log(s)
    if show_lambda and ax2 is not None:
        s_values = list(results)
        
        ax2.errorbar(s_values, lambdas, yerr=lambda_errors, fmt='ko', ls='--', capsize=3, label=r"$\lambda^2$ values")
        
        ax2.hlines(lambda_mean, min(s_values)/1.5, max(s_values)*1.5, colors='red', linestyles='-', label=r"Weighted Mean $\lambda^2$")
        
        ax2.fill_between([min(s_values)/1.5, max(s_values)*1.5], 
                         lambda_mean - err_mean, lambda_mean + err_mean, 
                         color='red', alpha=0.1, label=r"Mean Uncertainty ($1\sigma$)")

        ax2.set_xscale('log')
        ax2.grid(True, which="both", ls="-", alpha=0.3)
        ax2.set_xlabel(r"log(s)")
        ax2.set_ylabel(r"$\lambda^2$")
        ax2.set_title(rf"Dependence of $\lambda^2$ on Scale $s$")
        ax2.set_xlim(min(s_values)/1.5, max(s_values)*1.5)
        ax2.legend(loc='best')

    plt.tight_layout()
    plt.show()

def lambda_comp(lambdas,lambda_errors,s):
    '''
        Valuta la compatibilità dei lambda^2 ottenuti con una retta a pendenza nulla
    '''
    def line(x,m,b):
        return(m*x+b)
    popt,pcov = curve_fit(line,np.log(s),lambdas,[0,0.16],lambda_errors)
    m = popt[0]
    dm = np.sqrt(np.diag(pcov))[0]
    print(f"Fitting a line to Lambda^2, we get y=m log(s)+b with:\nm +/- dm = {m:.3f} +/- {dm:.3f}")


## Evaluating the quality of the fit through the Anderson–Darling test

The obtained fit can be evaluated in a way independent from the binning chosen to fit the PDF: through the Anderson–Darling test. 

It starts by evaluating the difference between the theoretical CDF ($F(x)$), in this case obtained from the Castaing function, and the experimental one ($F_n(x)$), a simple step function going from 0 to 1 with steps of $1/n$ (where $n$ is the length of the dataset). This difference, for each value of $x$, is squared and weighted by $w(x)$. 

For the Anderson-Darling test the weight function is $w(x)=[F(x)(1-F(x))]^{-1}$; this weight is particularly useful in our situation, since it values the tails of the CDF more: 

$$\text{as } x\rightarrow -\infty, \ F(x)\rightarrow 0 \quad \text{and} \quad \text{as } x\rightarrow +\infty, \ (1-F(x))\rightarrow 0$$

This is important since the tails are the regions where the fit to a normal function fails, due to the "fat tails" observed, and we expect the Castaing function to better represent the data.

The Anderson-Darling coefficient is given by:

$$A^2 = n \int \frac{(F_n(x)-F(x))^2}{F(x)(1-F(x))}dF(x)$$

Through discretization of the integral, we get the result:

$$A^2 = -n-S \quad , \text{with} \quad S = \sum^n_{i=1}\frac{2i-1}{n}[\ln(F(Y_i))+\ln(1-F(Y_{n-i+1}))]$$

where $Y_i$ is the i-th sorted data point.

Obtaining a significant value for this parameter isn't straightforward: common distributions have tabulated reference values that can be compared to the results obtained to determine whether or not the fit is representative of the dataset. In our case, we don't have any tabulated values to refer to: we had to create the table.

### The Monte Carlo Approach

After having obtained an optimal value for the parameter $\hat{\lambda}^2$ and for the coefficient $A_{obs}^2$, we had to generate a statistic for $A^2$ to evaluate the p-value of our result. 

Starting from the Castaing distribution with parameter $\hat{\lambda}^2$, we generated N different datasets. For every dataset, we repeated the fit operation, and thus we obtained a set of optimal parameters ${\hat{\lambda}^2_i}$. For each $\hat{\lambda}^2_i$, we calculated an $A_i^2$. The p-value was then calculated as the ratio between the number of $A_i^2$ bigger than $A_{obs}^2$ and the total number of $A_i^2$.

The pain didn't end here: for big samples, the Anderson–Darling test turned out to be very sensitive to the number of datapoints in the dataset; with $n\sim10^4$, the test tracks small differences between the sample and the theoretical curve, thus giving big $A^2$ values and vanishing p-values. The solution to this problem was ***subsampling***.
After calculating $A_{obs}^2$, we randomly selected subsets of size $M \sim 10^2$ from the original dataset and repeated the Monte Carlo procedure described above. This approach yielded more meaningful p-values, reflecting the physical validity of the model rather than statistical fluctuations due to sample size.

In [ ]:
results, lambdas, lambda_errors, a_s = get_res(df)

plot_results(results,lambdas,lambda_errors)

DA COMMENTARE CHE POTREBBERO ESSERCI PROBLEMI A SCALE PIU' BASSE SUI P-VALUES PER TREND LOCALI (RESPIRAZIONE,..) CHE VANNO A CREARE FLUTTUAZIONI CHE NON CONSIDERIAMO. QUANDO AUMENTIAMO S NON ABBIAMO IL PROBLEMA

In [ ]:
lambda_comp( lambdas,lambda_errors,list(results) )